In [1]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re, json, random
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, f1_score, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [3]:
LABELS = ["toxicity", "hate", "harassment", "abuse"]
JIGSAW_PATH      = "train.csv"
HASOC_TRAIN_PATH = "hindi_dataset.tsv"
HASOC_TEST_PATH  = "hasoc2019_hi_test_gold_2919.tsv"

MAX_LEN = 100                     
MIN_WORD_FREQ = 2                 
EMBED_DIM = 128
HIDDEN_DIM = 128
DROPOUT = 0.3
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
MAX_EPOCHS = 15
PATIENCE = 3                      

MODEL_PATH = "toxicity_model.pt"
VOCAB_PATH = "vocab.json"
CONFIG_PATH = "model_config.json"

In [4]:
jigsaw_df = pd.read_csv(JIGSAW_PATH)

jigsaw_df["toxicity"]   = ((jigsaw_df["toxic"] == 1) | (jigsaw_df["severe_toxic"] == 1)).astype(int)
jigsaw_df["hate"]       = jigsaw_df["identity_hate"].astype(int)
jigsaw_df["harassment"] = ((jigsaw_df["insult"] == 1) | (jigsaw_df["threat"] == 1)).astype(int)
jigsaw_df["abuse"]      = ((jigsaw_df["obscene"] == 1) | (jigsaw_df["severe_toxic"] == 1)).astype(int)

jigsaw_df = jigsaw_df.rename(columns={"comment_text": "text"})
jigsaw_df = jigsaw_df[["text"] + LABELS].dropna().drop_duplicates(subset="text")
jigsaw_df["label_mask"] = [[1, 1, 1, 1]] * len(jigsaw_df)   # all 4 labels known

print("jigsaw rows:", len(jigsaw_df))
print(jigsaw_df[LABELS].mean())

jigsaw rows: 159571
toxicity      0.095844
hate          0.008805
harassment    0.050435
abuse         0.053437
dtype: float64


In [5]:
hasoc_train = pd.read_csv(HASOC_TRAIN_PATH, sep="\t")
hasoc_test  = pd.read_csv(HASOC_TEST_PATH, sep="\t")
hasoc_df = pd.concat([hasoc_train, hasoc_test], ignore_index=True)

hasoc_df["toxicity"] = (hasoc_df["task_1"] == "HOF").astype(int)   
hasoc_df["hate"] = 0.0
hasoc_df["harassment"] = 0.0
hasoc_df["abuse"] = 0.0

hasoc_df = hasoc_df[["text"] + LABELS].dropna().drop_duplicates(subset="text")
hasoc_df["label_mask"] = [[1, 0, 0, 0]] * len(hasoc_df)   

print("hasoc rows:", len(hasoc_df))
print(hasoc_df["toxicity"].mean())

combined_df = pd.concat([jigsaw_df, hasoc_df], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("\ncombined rows:", len(combined_df))
print(combined_df[LABELS].mean())

hasoc rows: 5982
0.5138749582079573

combined rows: 165553
toxicity      0.110949
hate          0.008487
harassment    0.048613
abuse         0.051506
dtype: float64


In [7]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
WS_RE = re.compile(r"\s+")
def clean_text(text):
    text = text.lower()
    text = URL_RE.sub(" <URL> ", text)
    text = WS_RE.sub(" ", text).strip()
    return text
def tokenize(text):
    return text.split()

combined_df["tokens"] = combined_df["text"].apply(clean_text).apply(tokenize)
combined_df[["text", "tokens", "label_mask"]].head()

,text,tokens,label_mask
0,"your version looks like totally biased, may i ...","[your, version, looks, like, totally, biased,,...","[1, 1, 1, 1]"
1,Unless it is the first word in a sentence!,"[unless, it, is, the, first, word, in, a, sent...","[1, 1, 1, 1]"
2,"No, I'd guess you aren't have many African Ame...","[no,, i'd, guess, you, aren't, have, many, afr...","[1, 1, 1, 1]"
3,shovon is the vandal the only vandal here his ...,"[shovon, is, the, vandal, the, only, vandal, h...","[1, 1, 1, 1]"
4,Off-wiki? \nWhat does that mean?,"[off-wiki?, what, does, that, mean?]","[1, 1, 1, 1]"


In [8]:
train_df, temp_df = train_test_split(combined_df, train_size=0.70, random_state=SEED)
val_df, test_df = train_test_split(temp_df, train_size=0.5, random_state=SEED)
len(train_df), len(val_df), len(test_df)

(115887, 24833, 24833)

In [9]:
PAD, UNK = "<PAD>", "<UNK>"

def build_vocab(tokenized_texts, min_freq=MIN_WORD_FREQ):
    counter = Counter()
    for toks in tokenized_texts:
        counter.update(toks)
    vocab = {PAD: 0, UNK: 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_df["tokens"])
print("vocab size:", len(vocab))

def encode(tokens, vocab, max_len=MAX_LEN):
    ids = [vocab.get(t, vocab[UNK]) for t in tokens][:max_len]
    ids += [vocab[PAD]] * (max_len - len(ids))
    return ids

vocab size: 135392


In [10]:
class ToxicityDataset(Dataset):
    """Same as before, but also returns a per-row label_mask so the loss
    can ignore labels a given source dataset never provided."""
    def __init__(self, dframe, vocab):
        self.tokens = dframe["tokens"].tolist()
        self.labels = dframe[LABELS].values.astype("float32")
        self.masks = np.stack(dframe["label_mask"].values).astype("float32")
        self.vocab = vocab

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        ids = encode(self.tokens[idx], self.vocab)
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(self.labels[idx]),
            torch.tensor(self.masks[idx]),
        )

train_ds = ToxicityDataset(train_df, vocab)
val_ds   = ToxicityDataset(val_df, vocab)
test_ds  = ToxicityDataset(test_df, vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [11]:
class BiLSTMToxicityClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)   

    def forward(self, x):
        embedded = self.embedding(x)
        _, (h_n, _) = self.lstm(embedded)
        combined = torch.cat((h_n[-2], h_n[-1]), dim=1)     
        combined = self.dropout(combined)
        return self.fc(combined)                            

model = BiLSTMToxicityClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM, len(LABELS), DROPOUT).to(device)
model

BiLSTMToxicityClassifier(
  (embedding): Embedding(135392, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=4, bias=True)
)

In [12]:
def compute_pos_weight(dframe):
    weights = []
    for i, label in enumerate(LABELS):
        known = np.stack(dframe["label_mask"].values)[:, i] == 1
        sub = dframe[known]
        pos = sub[label].sum()
        neg = len(sub) - pos
        weights.append(neg / max(pos, 1))
    return torch.tensor(weights, dtype=torch.float32)

def masked_bce_loss(logits, targets, mask, pos_weight=None):
    per_element = nn.functional.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight, reduction="none"
    )
    per_element = per_element * mask
    return per_element.sum() / mask.sum().clamp(min=1.0)

pos_weight = compute_pos_weight(train_df).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_val_loss = float("inf")
patience_counter = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    train_loss = 0.0
    for x, y, m in train_loader:
        x, y, m = x.to(device), y.to(device), m.to(device)
        optimizer.zero_grad()
        loss = masked_bce_loss(model(x), y, m, pos_weight)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y, m in val_loader:
            x, y, m = x.to(device), y.to(device), m.to(device)
            val_loss += masked_bce_loss(model(x), y, m, pos_weight).item() * x.size(0)
    val_loss /= len(val_ds)

    print(f"epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)   
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(torch.load(MODEL_PATH))   


epoch 1: train_loss=0.7718 val_loss=0.6413
epoch 2: train_loss=0.4868 val_loss=0.5123
epoch 3: train_loss=0.3609 val_loss=0.5045
epoch 4: train_loss=0.2805 val_loss=0.6325
epoch 5: train_loss=0.2173 val_loss=0.7279
epoch 6: train_loss=0.1789 val_loss=0.7588
Early stopping.


<All keys matched successfully>

In [13]:
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_preds, all_true, all_masks = [], [], []
    with torch.no_grad():
        for x, y, m in loader:
            x = x.to(device)
            probs = torch.sigmoid(model(x)).cpu()
            all_preds.append((probs >= threshold).float())
            all_true.append(y)
            all_masks.append(m)
    return torch.cat(all_preds).numpy(), torch.cat(all_true).numpy(), torch.cat(all_masks).numpy()

y_pred, y_true, y_mask = evaluate(model, test_loader)

for i, label in enumerate(LABELS):
    valid = y_mask[:, i] == 1   
    p, r, f, _ = precision_recall_fscore_support(
        y_true[valid, i], y_pred[valid, i], average="binary", zero_division=0
    )
    cm = confusion_matrix(y_true[valid, i], y_pred[valid, i])
    fn = int(cm[1][0]) if cm.shape == (2, 2) else None
    print(f"{label:12s} n={valid.sum():>7d} precision={p:.3f} recall={r:.3f} f1={f:.3f} false_negatives={fn}")

overall_valid = y_mask[:, 0] == 1
print("\nmacro F1 (labels evaluated on their own valid rows only, see table above)")


toxicity     n=  24833 precision=0.555 recall=0.865 f1=0.676 false_negatives=379
hate         n=  23939 precision=0.084 recall=0.909 f1=0.153 false_negatives=19
harassment   n=  23939 precision=0.426 recall=0.876 f1=0.574 false_negatives=151
abuse        n=  23939 precision=0.524 recall=0.879 f1=0.656 false_negatives=159

macro F1 (labels evaluated on their own valid rows only, see table above)


In [16]:
def predict(text):
    tokens = tokenize(clean_text(text))
    ids = torch.tensor([encode(tokens, vocab)], dtype=torch.long).to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(ids))[0].cpu().tolist()
    return dict(zip(LABELS, probs))

for example in [
    "I really enjoyed this movie.",
    "You are a disgusting idiot.",
    "I don't think you're an idiot.",
    "xoxo ur so dum lol",
]:
    print(example, "->", predict(example))


I really enjoyed this movie. -> {'toxicity': 0.19232361018657684, 'hate': 0.06253554672002792, 'harassment': 0.05997922644019127, 'abuse': 0.07407082617282867}
You are a disgusting idiot. -> {'toxicity': 0.9975559711456299, 'hate': 0.982549786567688, 'harassment': 0.9942349791526794, 'abuse': 0.9948850274085999}
I don't think you're an idiot. -> {'toxicity': 0.9892947673797607, 'hate': 0.9413655400276184, 'harassment': 0.9669862985610962, 'abuse': 0.9709168672561646}
xoxo ur so dum lol -> {'toxicity': 0.3521280288696289, 'hate': 0.2181110382080078, 'harassment': 0.18400169909000397, 'abuse': 0.13927718997001648}


In [ ]:
with open(VOCAB_PATH, "w") as f:
    json.dump(vocab, f)

with open(CONFIG_PATH, "w") as f:
    json.dump({
        "vocab_size": len(vocab), "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM,
        "num_labels": len(LABELS), "dropout": DROPOUT, "max_len": MAX_LEN, "labels": LABELS,
    }, f)

print("Saved:", MODEL_PATH, VOCAB_PATH, CONFIG_PATH)
